# No.6 デノイジング — WNNM（発展・任意）

> ⚠️ **256×256画像で数分かかります。** まず `denoise_bm3d.ipynb` を完了してから取り組んでください。

WNNM（Weighted Nuclear Norm Minimization）を numpy で実装します。
BM3D と同様にパッチベースの手法ですが、パッチ群を低ランク行列として推定します。

実装は `wnnm_simple.py` にあります。

In [ ]:
import numpy as np
import skimage.io
import skimage.metrics
import matplotlib.pyplot as plt
import japanize_matplotlib
import time
import sys
sys.path.insert(0, '.')
from wnnm_simple import wnnm_denoise


In [ ]:
noise_sigma = 0.05

A  = skimage.io.imread('data/original.pgm').astype(float) / 255.0
rng = np.random.default_rng(seed=0)
An = np.clip(A + rng.normal(0, noise_sigma, A.shape), 0, 1)

print('WNNM デノイジング開始（数分かかります）...')
t0 = time.time()
denoised = wnnm_denoise(An, noise_sigma=noise_sigma, outer_iter=2)
elapsed = time.time() - t0
print(f'完了: {elapsed:.1f}秒')

In [ ]:
psnr_noisy = skimage.metrics.peak_signal_noise_ratio(A, An, data_range=1.0)
psnr       = skimage.metrics.peak_signal_noise_ratio(A, denoised, data_range=1.0)
ssim       = skimage.metrics.structural_similarity(A, denoised, data_range=1.0)
print(f'Noisy PSNR: {psnr_noisy:.2f} dB')
print(f'WNNM  PSNR: {psnr:.2f} dB,  SSIM: {ssim:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, im, title in zip(axes,
                          [A, An, denoised],
                          ['Original',
                           f'Noisy ({psnr_noisy:.1f}dB)',
                           f'WNNM ({psnr:.1f}dB, SSIM={ssim:.3f})']):
    ax.imshow(np.clip(im, 0, 1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
fig.tight_layout()
plt.show()